# Ahrefs growth test: HUM Nutrition

Ahrefs defines organic traffic as an estimated monthly count of clicks from Google, not total website visits. This notebook tests that metric for `humnutrition.com` and computes a simple growth rate from the Ahrefs history endpoint.

If you want me to run it end-to-end here, I need an Ahrefs key in the environment. The notebook accepts either `ahrefs_api_key` or `AHREFS_API_KEY` from `.env`.

In [ ]:
from __future__ import annotations

import os
from datetime import date, timedelta
from pathlib import Path

import requests

TARGET = "humnutrition.com"
BASE_URL = "https://api.ahrefs.com/v3/site-explorer"


def load_env_value(*names: str) -> str:
    for name in names:
        value = os.getenv(name, "").strip()
        if value:
            return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            if key.strip() in names:
                cleaned = value.strip().strip('"').strip("'")
                if cleaned:
                    return cleaned

    return ""


API_KEY = load_env_value("ahrefs_api_key", "AHREFS_API_KEY")

if not API_KEY:
    raise SystemExit("Set ahrefs_api_key or AHREFS_API_KEY in .env before running this notebook.")

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json",
}


def ahrefs_get(endpoint: str, params: dict[str, str]) -> dict:
    response = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def safe_pct(current: float | int | None, previous: float | int | None) -> float | None:
    if previous in (None, 0):
        return None
    if current is None:
        return None
    return ((float(current) - float(previous)) / float(previous)) * 100.0


today = date.today()
window_start = today - timedelta(days=60)
window_midpoint = today - timedelta(days=30)

print(f"Loaded Ahrefs key: {'yes' if API_KEY else 'no'}")
print(f"Target: {TARGET}")
print(f"Window: {window_start.isoformat()} -> {today.isoformat()}")

In [ ]:
# Snapshot: estimated monthly organic and paid search traffic.
metrics = ahrefs_get(
    "metrics",
    {
        "target": TARGET,
        "mode": "domain",
        "protocol": "both",
        "traffic_mode": "static",
        "volume_mode": "monthly",
        "output": "json",
    },
)

metrics

In [ ]:
# History: daily estimated organic traffic so we can compute a simple growth rate.
history = ahrefs_get(
    "metrics-history",
    {
        "target": TARGET,
        "mode": "domain",
        "protocol": "both",
        "date_from": window_start.isoformat(),
        "date_to": today.isoformat(),
        "history_grouping": "daily",
        "traffic_mode": "static",
        "volume_mode": "monthly",
        "output": "json",
    },
)

series = history.get("metrics", [])
series = sorted(series, key=lambda row: row.get("date", ""))

if not series:
    raise RuntimeError("Ahrefs returned no history rows for this target.")

first_row = series[0]
mid_row = min(series, key=lambda row: abs((date.fromisoformat(row["date"]) - window_midpoint).days))
last_row = series[-1]

first_traffic = first_row.get("org_traffic")
mid_traffic = mid_row.get("org_traffic")
last_traffic = last_row.get("org_traffic")

growth_30d = safe_pct(last_traffic, mid_traffic)
growth_60d = safe_pct(last_traffic, first_traffic)

summary = {
    "first_date": first_row.get("date"),
    "mid_date": mid_row.get("date"),
    "last_date": last_row.get("date"),
    "first_org_traffic": first_traffic,
    "mid_org_traffic": mid_traffic,
    "last_org_traffic": last_traffic,
    "growth_30d_pct": round(growth_30d, 2) if growth_30d is not None else None,
    "growth_60d_pct": round(growth_60d, 2) if growth_60d is not None else None,
}

summary

In [ ]:
# Compact readout for the drill-down definition.
org_traffic = metrics.get("metrics", {}).get("org_traffic")
paid_traffic = metrics.get("metrics", {}).get("paid_traffic")

print(f"HUM Nutrition estimated organic search traffic: {org_traffic}")
print(f"HUM Nutrition estimated paid search traffic: {paid_traffic}")
print(f"30-day organic traffic growth: {summary['growth_30d_pct']}%")
print(f"60-day organic traffic growth: {summary['growth_60d_pct']}%")
print("Note: Ahrefs traffic is an estimate of monthly search visitors, not total website visitors.")